# Chess Scoresheet Cell Extractor - Kaggle Pipeline Demo

Notebook này tái hiện pipeline hiện tại của repo `chess-scoresheet-cell-extractor` trên Kaggle.

Mục tiêu:

- Clone code và dữ liệu input trực tiếp từ GitHub, không dùng dữ liệu upload thủ công trong `/kaggle/input`.
- Chọn một ảnh scoresheet trong `inputs/` để minh họa từng bước xử lý.
- Chạy lại pipeline đầy đủ để sinh crop cell vào thư mục output của notebook.

Trên Kaggle, cần bật `Internet` trong phần notebook settings trước khi chạy cell clone repo.

## 1. Clone repo hiện tại từ GitHub

Cell này tải toàn bộ code, `requirements.txt`, ảnh `inputs/` và output mẫu đã được commit lên GitHub. Notebook không yêu cầu upload input mới lên Kaggle.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/chess-scoresheet-cell-extractor.git'
REPO_NAME = 'chess-scoresheet-cell-extractor'

CURRENT_DIR = Path.cwd().resolve()
USING_LOCAL_REPO = (CURRENT_DIR / 'extract_cells.py').exists()

if USING_LOCAL_REPO:
    REPO_DIR = CURRENT_DIR
    WORK_DIR = REPO_DIR / '_kaggle_working'
else:
    WORK_DIR = Path('/kaggle/working')
    if not WORK_DIR.exists():
        WORK_DIR = CURRENT_DIR / '_kaggle_working'
    REPO_DIR = WORK_DIR / REPO_NAME

WORK_DIR.mkdir(parents=True, exist_ok=True)

if USING_LOCAL_REPO:
    print(f'Using local repo: {REPO_DIR}')
elif (REPO_DIR / '.git').exists():
    print(f'Repo already exists: {REPO_DIR}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repository. Remove it or choose another folder.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

if (REPO_DIR / '.git').exists():
    commit = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
        text=True,
    ).strip()
else:
    commit = 'local files without .git metadata'

print(f'Working directory: {WORK_DIR}')
print(f'Repository: {REPO_DIR}')
print(f'Commit/source: {commit}')


## 2. Cài dependency và import pipeline

Pipeline chính nằm trong `extract_cells.py`. Notebook import trực tiếp các hàm từ file này để phần demo bám đúng implementation hiện tại.

In [ ]:
import csv
import importlib
import math
import shutil

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')],
    check=True,
)

sys.path.insert(0, str(REPO_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import extract_cells as pipeline

pipeline = importlib.reload(pipeline)
plt.rcParams['figure.dpi'] = 120

def show_image(ax, image_data, title, cmap=None):
    ax.imshow(image_data, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')


def show_cell_gallery(cell_items, cols=4, max_items=16):
    selected = cell_items if max_items is None else cell_items[:max_items]
    if not selected:
        print('No cells to show')
        return

    rows = math.ceil(len(selected) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 1.8))
    axes = np.array(axes).reshape(-1)

    for ax, item in zip(axes, selected):
        ax.imshow(item['image'], cmap='gray')
        ax.set_title(item['file_name'], fontsize=7)
        ax.axis('off')

    for ax in axes[len(selected):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()


def cutoff_label(cutoff_index):
    return 'None' if cutoff_index is None else str(cutoff_index + 1)


print(f'OpenCV: {cv2.__version__}')
print(f'NumPy: {np.__version__}')


## 3. Chọn một ảnh input để demo

Notebook dùng ảnh `012_0.png` trong repo làm ảnh minh họa. Đây là một scoresheet có đủ grid rõ để demo các bước: threshold, tách đường kẻ, tìm contour, sort cell, crop cell và lọc ô có chữ viết tay.

In [ ]:
INPUT_DIR = REPO_DIR / 'inputs'
COMMITTED_OUTPUT_DIR = REPO_DIR / 'outputs'

input_images = sorted(INPUT_DIR.glob('*.png')) + sorted(INPUT_DIR.glob('*.jpg')) + sorted(INPUT_DIR.glob('*.jpeg'))
print('Input images:')
for path in input_images:
    print(f'- {path.name}')

DEMO_IMAGE_NAME = '012_0.png'
demo_path = INPUT_DIR / DEMO_IMAGE_NAME
if not demo_path.exists():
    raise FileNotFoundError(f'Demo image not found: {demo_path}')

image = Image.open(demo_path).convert('RGB')
print(f'Demo image: {demo_path}')
print(f'Size: {image.width} x {image.height}')

fig, ax = plt.subplots(figsize=(8, 10))
show_image(ax, np.array(image), f'Original image - {DEMO_IMAGE_NAME}')
plt.show()

## 4. Chuyển sang grayscale và nhị phân hóa bằng Otsu

Pipeline không OCR trực tiếp trên ảnh gốc. Đầu tiên ảnh được chuyển sang grayscale, sau đó dùng Otsu threshold để tách vùng sáng/tối. Ảnh nhị phân này là đầu vào cho bước tìm đường kẻ bảng.

In [ ]:
gray_image = image.convert('L')
binary_image = pipeline.image_to_binary(gray_image)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_image(axes[0], np.array(image), 'Original')
show_image(axes[1], np.array(gray_image), 'Grayscale', cmap='gray')
show_image(axes[2], binary_image, 'Otsu binary', cmap='gray')
plt.tight_layout()
plt.show()

## 5. Tách đường kẻ ngang/dọc của bảng

Từ ảnh nhị phân, pipeline đảo màu rồi dùng morphology với kernel ngang và dọc. Kết quả là mask chứa các đường grid chính của scoresheet.

In [ ]:
inverted_binary = cv2.bitwise_not(binary_image)

horizontal_kernel_length = max(1, inverted_binary.shape[1] // 40)
vertical_kernel_length = max(1, inverted_binary.shape[0] // 40)

horizontal_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT, (horizontal_kernel_length, 1)
)
vertical_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT, (1, vertical_kernel_length)
)

horizontal_lines = cv2.erode(inverted_binary, horizontal_kernel, iterations=1)
horizontal_lines = cv2.dilate(horizontal_lines, horizontal_kernel, iterations=1)

vertical_lines = cv2.erode(inverted_binary, vertical_kernel, iterations=1)
vertical_lines = cv2.dilate(vertical_lines, vertical_kernel, iterations=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show_image(axes[0], horizontal_lines, 'Horizontal lines only', cmap='gray')
show_image(axes[1], vertical_lines, 'Vertical lines only', cmap='gray')
plt.tight_layout()
plt.show()

In [ ]:
grid_lines = cv2.add(horizontal_lines, vertical_lines)

grid_overlay = np.array(image).copy()
grid_overlay[grid_lines > 0] = [255, 0, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_image(axes[0], binary_image, 'Binary input', cmap='gray')
show_image(axes[1], grid_lines, 'Detected grid lines', cmap='gray')
show_image(axes[2], grid_overlay, 'Grid overlay on original')
plt.tight_layout()
plt.show()

## 6. Tìm contour ô và sắp xếp theo thứ tự scoresheet

Pipeline tìm contour hình tứ giác từ grid mask, lọc theo diện tích gần median để loại nhiễu, sau đó gán cột và hàng. Thứ tự output là nửa trái trước, nửa phải sau; trong mỗi nước đi, ô White đứng trước ô Black.

In [ ]:
contours = pipeline.find_move_cell_contours(grid_lines)
detected_cells = pipeline.sort_cells_by_scoresheet_move_order(contours)
cells = pipeline.complete_scoresheet_cells(detected_cells, gray_image.size)

contour_overlay = np.array(image).copy()
for index, cell_info in enumerate(cells):
    contour = np.asarray(cell_info['contour'], dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(contour_overlay, [contour], True, (255, 0, 0), 3)
    if index < 12:
        x_min, y_min, _, _ = pipeline.contour_bounds(cell_info['contour'])
        cv2.putText(
            contour_overlay,
            str(index + 1),
            (x_min + 3, y_min + 18),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2,
            cv2.LINE_AA,
        )

print(f'Raw rectangle contours: {len(contours)}')
print(f'Sorted detected cells: {len(detected_cells)}')
print(f'Cells used for output after reconstruction: {len(cells)}')

fig, ax = plt.subplots(figsize=(8, 10))
show_image(ax, contour_overlay, 'Move-cell grid used by pipeline')
plt.show()


## 7. Separate Saved Crop From Detection Crop

Saved crops keep the larger OCR padding. The non-empty decision uses a separate detection crop: vertically tighter, based on the original cell bounds, and shifted past the printed move-number area for White columns.


In [ ]:
CELL_TOP_PADDING_RATIO = pipeline.CELL_TOP_PADDING_RATIO
CELL_BOTTOM_PADDING_RATIO = pipeline.CELL_BOTTOM_PADDING_RATIO
TRIM_NUMBER_COLUMN_RATIO = pipeline.TRIM_NUMBER_COLUMN_RATIO
MIN_DARK_RATIO = pipeline.MIN_DARK_RATIO
DETECTION_VERTICAL_MARGIN_RATIO = pipeline.DETECTION_VERTICAL_MARGIN_RATIO
DETECTION_HORIZONTAL_MARGIN_RATIO = pipeline.DETECTION_HORIZONTAL_MARGIN_RATIO
STOP_AFTER_FIRST_EMPTY = pipeline.STOP_AFTER_FIRST_EMPTY
EMPTY_LOOKAHEAD = pipeline.EMPTY_LOOKAHEAD

cropped_cells = []
for index, cell_info in enumerate(cells):
    save_crop = pipeline.crop_cell(
        gray_image=gray_image,
        contour=cell_info['contour'],
        top_padding_ratio=CELL_TOP_PADDING_RATIO,
        bottom_padding_ratio=CELL_BOTTOM_PADDING_RATIO,
    )
    save_crop = pipeline.trim_printed_move_number_column(
        image=save_crop,
        column=cell_info['column'],
        trim_ratio=TRIM_NUMBER_COLUMN_RATIO,
    )

    detection_crop, detection_bbox = pipeline.crop_cell_detection_region(
        gray_image=gray_image,
        contour=cell_info['contour'],
        column=cell_info['column'],
        trim_number_column_ratio=TRIM_NUMBER_COLUMN_RATIO,
        detection_vertical_margin_ratio=DETECTION_VERTICAL_MARGIN_RATIO,
        detection_horizontal_margin_ratio=DETECTION_HORIZONTAL_MARGIN_RATIO,
    )
    debug = pipeline.handwriting_ink_analysis(detection_crop, MIN_DARK_RATIO)

    cropped_cells.append(
        {
            'index': index,
            'file_name': pipeline.move_cell_name(demo_path.stem, index, len(cells)),
            'image': save_crop,
            'detection_image': detection_crop,
            'detection_bbox': detection_bbox,
            'debug': debug,
            'ink_ratio': float(debug['ink_ratio']),
            'raw_non_empty': bool(debug['non_empty']),
            'info': cell_info,
        }
    )

print(f'Cropped cells for save/OCR: {len(cropped_cells)}')
print(f'Detection vertical margin ratio: {DETECTION_VERTICAL_MARGIN_RATIO}')
print(f'Detection horizontal margin ratio: {DETECTION_HORIZONTAL_MARGIN_RATIO}')
print(f'Stop after first empty: {STOP_AFTER_FIRST_EMPTY}, empty lookahead: {EMPTY_LOOKAHEAD}')
show_cell_gallery(cropped_cells, cols=4, max_items=16)


In [ ]:
if not cropped_cells:
    raise RuntimeError('No cropped cells to show')

example_crop = next(
    (item for item in cropped_cells if item['info'].get('column') in (1, 3)),
    cropped_cells[0],
)
example_info = example_crop['info']
example_contour = np.asarray(example_info['contour'], dtype=np.int32).reshape((-1, 1, 2))
x_min, y_min, x_max, y_max = pipeline.contour_bounds(example_info['contour'])
cell_width = x_max - x_min
cell_height = y_max - y_min

crop_left = max(0, x_min)
crop_top = max(0, y_min - int(cell_height * CELL_TOP_PADDING_RATIO))
crop_right = min(image.width, x_max)
crop_bottom = min(image.height, y_max + int(cell_height * CELL_BOTTOM_PADDING_RATIO))

detect_left, detect_top, detect_right, detect_bottom = example_crop['detection_bbox']

before_crop_overlay = np.array(image).copy()
cv2.rectangle(
    before_crop_overlay,
    (crop_left, crop_top),
    (crop_right, crop_bottom),
    (255, 0, 0),
    4,
)
cv2.rectangle(
    before_crop_overlay,
    (detect_left, detect_top),
    (detect_right, detect_bottom),
    (0, 180, 0),
    4,
)
cv2.polylines(before_crop_overlay, [example_contour], True, (0, 0, 255), 3)

zoom_margin_x = int(cell_width * 0.6)
zoom_margin_y = int(cell_height * 1.2)
zoom_left = max(0, crop_left - zoom_margin_x)
zoom_top = max(0, crop_top - zoom_margin_y)
zoom_right = min(image.width, crop_right + zoom_margin_x)
zoom_bottom = min(image.height, crop_bottom + zoom_margin_y)
before_crop_zoom = before_crop_overlay[zoom_top:zoom_bottom, zoom_left:zoom_right]

after_crop = np.array(example_crop['image'])
detection_crop = np.array(example_crop['detection_image'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
show_image(axes[0], before_crop_zoom, f"Red save crop, green detect crop\n{example_crop['file_name']}")
show_image(axes[1], after_crop, 'Saved/OCR crop keeps padding', cmap='gray')
show_image(axes[2], detection_crop, 'Detection crop is vertically tighter', cmap='gray')
plt.tight_layout()
plt.show()


## 8. Non-Empty Filtering With Continuous-Move Cutoff

`raw_non_empty` is computed on the detection crop. When `stop_after_first_empty` is enabled, the pipeline saves only the scoresheet-order prefix before the first confirmed empty cell; later notes or results are rejected even when they contain ink.


In [ ]:
raw_flags = [item['raw_non_empty'] for item in cropped_cells]
cutoff_index = None
if STOP_AFTER_FIRST_EMPTY:
    cutoff_index = pipeline.find_first_confirmed_empty_index(raw_flags, EMPTY_LOOKAHEAD)

for item in cropped_cells:
    is_cutoff_cell = cutoff_index is not None and item['index'] == cutoff_index
    after_cutoff = cutoff_index is not None and item['index'] > cutoff_index
    if STOP_AFTER_FIRST_EMPTY:
        keep = cutoff_index is None or item['index'] < cutoff_index
    else:
        keep = item['raw_non_empty']

    if keep:
        reject_reason = ''
    elif is_cutoff_cell:
        reject_reason = 'blank'
    elif after_cutoff:
        reject_reason = 'after_cutoff'
    else:
        reject_reason = 'blank'

    item['after_cutoff'] = after_cutoff
    item['keep'] = keep
    item['reject_reason'] = reject_reason

kept_cells = [item for item in cropped_cells if item['keep']]
blank_rejected_cells = [item for item in cropped_cells if item['reject_reason'] == 'blank']
after_cutoff_cells = [item for item in cropped_cells if item['reject_reason'] == 'after_cutoff']
raw_non_empty_cells = [item for item in cropped_cells if item['raw_non_empty']]

print(f'Raw non-empty cells before cutoff logic: {len(raw_non_empty_cells)}')
print(f'Cutoff cell index: {cutoff_label(cutoff_index)}')
print(f'Cells saved after filtering: {len(kept_cells)}')
print(f'Cells rejected as blank: {len(blank_rejected_cells)}')
print(f'Cells rejected after cutoff: {len(after_cutoff_cells)}')

examples = []
if kept_cells:
    examples.append(kept_cells[-1])
if after_cutoff_cells:
    examples.append(after_cutoff_cells[0])
elif blank_rejected_cells:
    examples.append(blank_rejected_cells[0])

fig, axes = plt.subplots(len(examples), 6, figsize=(18, 3.2 * len(examples)))
axes = np.array(axes).reshape(len(examples), 6)

for row, item in enumerate(examples):
    debug = item['debug']
    state = 'save' if item['keep'] else item['reject_reason']
    title_prefix = f"{state}, ratio={item['ink_ratio']:.4f}"

    show_image(axes[row, 0], item['image'], f"Saved crop\n{item['file_name']}", cmap='gray')
    show_image(axes[row, 1], item['detection_image'], f'{title_prefix}\nDetection crop', cmap='gray')
    show_image(axes[row, 2], debug['roi'], 'ROI checked', cmap='gray')
    show_image(axes[row, 3], debug['contrast_ink'], 'Contrast ink', cmap='gray')
    show_image(axes[row, 4], debug['without_grid'], 'After grid removal', cmap='gray')
    show_image(axes[row, 5], debug['final_ink'], 'Final ink mask', cmap='gray')

plt.tight_layout()
plt.show()


## 9. Save Demo Results With Pipeline Diagnostics


In [ ]:
DEMO_OUTPUT_DIR = WORK_DIR / 'demo_pipeline_output'
if DEMO_OUTPUT_DIR.exists():
    shutil.rmtree(DEMO_OUTPUT_DIR)

manifest_path = DEMO_OUTPUT_DIR / demo_path.stem / f'{demo_path.stem}_manifest.csv'
result = pipeline.extract_cells_from_image_with_diagnostics(
    image_path=demo_path,
    output_dir=DEMO_OUTPUT_DIR,
    non_empty_only=True,
    min_dark_ratio=MIN_DARK_RATIO,
    trim_number_column_ratio=TRIM_NUMBER_COLUMN_RATIO,
    cell_top_padding_ratio=CELL_TOP_PADDING_RATIO,
    cell_bottom_padding_ratio=CELL_BOTTOM_PADDING_RATIO,
    save_debug=True,
    detection_vertical_margin_ratio=DETECTION_VERTICAL_MARGIN_RATIO,
    detection_horizontal_margin_ratio=DETECTION_HORIZONTAL_MARGIN_RATIO,
    stop_after_first_empty=STOP_AFTER_FIRST_EMPTY,
    empty_lookahead=EMPTY_LOOKAHEAD,
    manifest_path=manifest_path,
)

demo_generated_files = sorted((DEMO_OUTPUT_DIR / demo_path.stem).glob('*.png'))
print(f'Demo image: {DEMO_IMAGE_NAME}')
print(f'Contours/cells detected: contours={result.contour_count}, detected_cells={result.detected_cell_count}, output_cells={result.output_cell_count}')
print(f'Raw non-empty before cutoff: {result.raw_non_empty_count}')
print(f'Cutoff cell index: {cutoff_label(result.cutoff_index)}')
print(f'Saved cells: {result.saved_count}')
print(f'Rejected as blank: {result.blank_rejected_count}')
print(f'Rejected after cutoff: {result.after_cutoff_rejected_count}')
print(f'Manifest CSV: {manifest_path}')
print(f'Demo output directory: {DEMO_OUTPUT_DIR / demo_path.stem}')

generated_items = [
    {'file_name': path.name, 'image': Image.open(path).convert('L')}
    for path in demo_generated_files
]
show_cell_gallery(generated_items, cols=5, max_items=20)


## 10. Run the Full Pipeline on All Input Images and Write Debug Manifests


In [ ]:
FULL_OUTPUT_DIR = WORK_DIR / 'full_pipeline_output'
if FULL_OUTPUT_DIR.exists():
    shutil.rmtree(FULL_OUTPUT_DIR)

target_images = input_images
if not target_images:
    raise FileNotFoundError(f'No input images found in {INPUT_DIR}')

print('Running full pipeline with non_empty_only + continuous cutoff')
print(f'Output directory: {FULL_OUTPUT_DIR}')
print(f'Input images to process: {len(target_images)}')
run_summaries = []
combined_records = []
for image_path in target_images:
    manifest_path = FULL_OUTPUT_DIR / image_path.stem / f'{image_path.stem}_manifest.csv'
    result = pipeline.extract_cells_from_image_with_diagnostics(
        image_path=image_path,
        output_dir=FULL_OUTPUT_DIR,
        non_empty_only=True,
        min_dark_ratio=MIN_DARK_RATIO,
        trim_number_column_ratio=TRIM_NUMBER_COLUMN_RATIO,
        cell_top_padding_ratio=CELL_TOP_PADDING_RATIO,
        cell_bottom_padding_ratio=CELL_BOTTOM_PADDING_RATIO,
        save_debug=pipeline.SAVE_DEBUG,
        detection_vertical_margin_ratio=DETECTION_VERTICAL_MARGIN_RATIO,
        detection_horizontal_margin_ratio=DETECTION_HORIZONTAL_MARGIN_RATIO,
        stop_after_first_empty=STOP_AFTER_FIRST_EMPTY,
        empty_lookahead=EMPTY_LOOKAHEAD,
        manifest_path=manifest_path,
    )
    cell_files = sorted((FULL_OUTPUT_DIR / image_path.stem).glob(f'{image_path.stem}_cell_*.png'))
    file_count = len(cell_files)
    run_summaries.append((image_path, result, cell_files, manifest_path))
    combined_records.extend(result.manifest_records)
    print(
        f'{image_path.name}: contours={result.contour_count}, '
        f'detected_cells={result.detected_cell_count}, output_cells={result.output_cell_count}, '
        f'raw_non_empty={result.raw_non_empty_count}, cutoff={cutoff_label(result.cutoff_index)}, '
        f'saved={result.saved_count}, files={file_count}, '
        f'blank_rejected={result.blank_rejected_count}, after_cutoff_rejected={result.after_cutoff_rejected_count}, '
        f'manifest={manifest_path.name}'
    )

combined_manifest_path = FULL_OUTPUT_DIR / 'manifest_all.csv'
pipeline.write_manifest_csv(combined_manifest_path, combined_records)

all_cell_files = [cell_file for _, _, cell_files, _ in run_summaries for cell_file in cell_files]
print(f'Generated cell PNG files: {len(all_cell_files)}')
print(f'Combined manifest CSV: {combined_manifest_path}')

for image_path, result, cell_files, manifest_path in run_summaries:
    original_image = Image.open(image_path).convert('RGB')
    print('\n' + '=' * 80)
    print(f'Input image: {image_path.name}')
    print(f'Size: {original_image.width} x {original_image.height}')
    print(f'Extracted cells shown below: {len(cell_files)}')
    print(f'Manifest CSV: {manifest_path}')

    fig, ax = plt.subplots(figsize=(8, 10))
    show_image(ax, np.array(original_image), f'Original input - {image_path.name}')
    plt.show()

    cell_items = [
        {'file_name': cell_file.name, 'image': Image.open(cell_file).convert('L')}
        for cell_file in cell_files
    ]
    show_cell_gallery(cell_items, cols=6, max_items=None)
